In [ ]:
from google.colab import files
files.upload()   # upload kaggle.json


In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
!kaggle datasets download -d harshithabhaskaran/Echonet -p ./datasets/EchoNet --unzip


In [ ]:
!7z x ./datasets/EchoNet/EchoNet-Dynamic.7z.001 -o./Processed/EchoNet/


In [ ]:
# --- Standard Library ---
import math
import os
import random
import shutil

# --- Third-Party Libraries ---
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
from scipy.interpolate import splprep, splev
from scipy.ndimage import binary_closing, binary_dilation, binary_fill_holes, gaussian_filter
from skimage.morphology import disk
from scipy.spatial import ConvexHull

In [ ]:
# -----------------------------
# Setup paths for Colab (Kaggle dataset downloaded locally)
# -----------------------------
import os

# True if running in Colab
colab = True

if colab:
    # Path where you downloaded and unzipped EchoNet-Dynamic
    echo_net_dynamic_folder = '/content/Processed/EchoNet/EchoNet-Dynamic'
    output_dir = '/content/echonet_processed/'
else:
    # Local machine path
    echo_net_dynamic_folder = '../../data/echo-net-dynamic/'
    output_dir = '../../data/echo-net-dynamic-processed/'

# Frames and masks output folders
frames_dir = os.path.join(output_dir, 'frames')
output_mask_dir = os.path.join(output_dir, 'masks')
output_image_folder = frames_dir

# CSV paths
csv_path_mask = os.path.join(echo_net_dynamic_folder, 'VolumeTracings.csv')
csv_path_train_test_val = os.path.join(echo_net_dynamic_folder, 'FileList.csv')

# Video input folder
input_video_folder = os.path.join(echo_net_dynamic_folder, 'Videos')

# Create directories if they don't exist
os.makedirs(output_dir, exist_ok=True)
os.makedirs(frames_dir, exist_ok=True)
os.makedirs(output_mask_dir, exist_ok=True)
os.makedirs(output_image_folder, exist_ok=True)

print("✅ Paths set up for Colab")
print("Videos folder:", input_video_folder)
print("Masks output folder:", output_mask_dir)
print("Frames output folder:", output_image_folder)


In [ ]:
df_mask = pd.read_csv(csv_path_mask)
display(df_mask)

In [ ]:
df_train_test_val = pd.read_csv(csv_path_train_test_val)
display(df_train_test_val)

In [ ]:

split_ratios = df_train_test_val['Split'].value_counts(normalize=True)

split_ratios_percent = split_ratios * 100

print("Split ratios (as fractions):")
print(split_ratios)

print("\nSplit ratios (as percentages):")
print(split_ratios_percent)

each clip has 2 anotated images: end-diastole and end-systole

In [ ]:
# Get unique (FileName, Frame) pairs
df=df_mask.copy()
df_mask_filename_framenumber = df[['FileName', 'Frame']].drop_duplicates()

# Reset index (optional)
df_mask_filename_framenumber = df_mask_filename_framenumber.reset_index(drop=True)

print(df_mask_filename_framenumber)

In [ ]:
existing_videos = set(os.listdir(input_video_folder))
df = df[df['FileName'].isin(existing_videos)]
print("Remaining rows after filtering:", len(df))


In [ ]:
df = df.drop_duplicates(subset=['FileName', 'Frame'])


In [ ]:
for index, row in df.iterrows():
    filename = row['FileName']
    frame_num = int(row['Frame'])
    video_path = os.path.join(input_video_folder, filename)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Failed to open: {video_path}")
        continue

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
    ret, frame = cap.read()
    if ret:
        output_filename = f"{os.path.splitext(filename)[0]}_{frame_num}.png"
        output_path = os.path.join(output_image_folder, output_filename)
        cv2.imwrite(output_path, frame)
        print(f"Saved frame {frame_num} from {filename} to {output_path}")

    cap.release()


In [ ]:
# Set random seed for reproducibility
random.seed(42)

# List all PNG files in the folder
image_files = [f for f in os.listdir(output_image_folder) if f.endswith('.png')]

# Select 8 random images
selected_files = random.sample(image_files, 8)

# Plot the images in a 2x4 grid
plt.figure(figsize=(16, 8))
for i, file_name in enumerate(selected_files):
    img_path = os.path.join(output_image_folder, file_name)
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.subplot(2, 4, i + 1)
    plt.imshow(img_rgb)
    plt.title(file_name, fontsize=8)
    plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
shutil.make_archive(output_image_folder, 'zip', output_image_folder)

In [ ]:
dfa=df_mask.copy()
# Create the new dataframe with transformed columns
df_mask_list_format = pd.DataFrame()
df_mask_list_format['FileName'] = dfa['FileName'].str.replace('.avi', '', regex=False) + '_' + dfa['Frame'].astype(str) + '.png'
df_mask_list_format['MaskCoordinates'] = list(zip(zip(dfa['X1'], dfa['Y1']), zip(dfa['X2'], dfa['Y2'])))

# Display the result
display(df_mask_list_format)


In [ ]:
# Group by FileName and aggregate MaskCoordinates
df_mask_list_format_merged = df_mask_list_format.groupby('FileName', as_index=False)['MaskCoordinates'].agg(list)

# Flatten the list of coordinates for each FileName
df_mask_list_format_merged['MaskCoordinates'] = df_mask_list_format_merged['MaskCoordinates'].apply(
    lambda coord_lists: [coord for sublist in coord_lists for coord in sublist]
)


# Display the result
display(df_mask_list_format_merged)

print(df_mask_list_format_merged['MaskCoordinates'][0])


In [ ]:
print(df_mask_list_format_merged.shape)


In [ ]:
print(df_mask_list_format_merged.columns)


In [ ]:
print(df_mask_list_format_merged['MaskCoordinates'].head())


In [ ]:
for idx, row in tqdm(df_mask_list_format_merged.iterrows(), total=len(df_mask_list_format_merged)):
    file_name = row['FileName']

    # Force PNG extension
    if not file_name.lower().endswith('.png'):
        file_name = os.path.splitext(file_name)[0] + '.png'

    coords = row['MaskCoordinates']
    mask_img = generate_mask_image(coords, width, height)

    # Save mask
    output_path = os.path.join(output_mask_dir, file_name)
    Image.fromarray(mask_img * 255).save(output_path)


# Methods for saving generated masks

# Ploting masks

In [ ]:
plot_mask_images(df_mask_list_format_merged, count=16, seed=42)


In [ ]:
df_mask_list_format = pd.DataFrame()
df_mask_list_format['FileName'] = dfa['FileName'].str.replace('.avi', '', regex=False) + '_' + dfa['Frame'].astype(str) + '.png'
df_mask_list_format['MaskCoordinates'] = list(zip(zip(dfa['X1'], dfa['Y1']), zip(dfa['X2'], dfa['Y2'])))

# Merge coordinates for same frame
df_mask_list_format_merged = df_mask_list_format.groupby('FileName', as_index=False)['MaskCoordinates'].agg(list)
df_mask_list_format_merged['MaskCoordinates'] = df_mask_list_format_merged['MaskCoordinates'].apply(
    lambda coord_lists: [coord for sublist in coord_lists for coord in sublist]
)

# Check
df_mask_list_format_merged['FileName'].head(10)


# Plot mask and images side by side

In [ ]:
def plot_mask_images(df, count=12, seed=42):
    """
    Plot `count` randomly selected mask images from the dataframe,
    automatically adjusting the number of rows based on 4 columns per row,
    with each subplot having a dark blue background.
    """
    random.seed(seed)
    selected_rows = df.sample(n=count, random_state=seed).reset_index(drop=True)

    cols = 4
    rows = math.ceil(count / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3 * rows))
    axes = axes.flatten() if count > 1 else [axes]

    for i in range(rows * cols):
        ax = axes[i]
        ax.set_facecolor('#001f3f')  # Dark blue background
        if i < count:
            row = selected_rows.iloc[i]
            file_name = row['FileName'].replace('.avi', '.png')
            coords = row['MaskCoordinates']
            mask_img = generate_mask_image(coords, width, height)
            ax.imshow(mask_img, cmap='gray')
            ax.set_title(f'{file_name}', color='white')  # White text for contrast
        else:
            ax.axis('off')
        ax.axis('off')

    fig.patch.set_facecolor('#001f3f')  # Optional: set figure background to match
    plt.tight_layout()
    plt.show()

In [ ]:
# Example: plot 12 random mask images
plot_mask_images(df_mask_list_format_merged, count=12, seed=42)


In [ ]:
available_frames = set(os.listdir(frames_dir))
print("Total frames in folder:", len(available_frames))


In [ ]:
df_existing_masks = df_mask_list_format_merged[df_mask_list_format_merged['FileName'].isin(available_frames)]
print(f"Frames available for plotting: {len(df_existing_masks)}")


# Show some of masks before saving them

In [ ]:
def plot_combined_mask_overlay_fixed(df, image_folder, count=8, seed=42):
    """
    Plot `count` samples with Original Image, Mask, Overlay.
    Uses images from image_folder and masks from df.
    """
    import random
    import matplotlib.pyplot as plt
    import cv2
    import numpy as np
    from scipy.ndimage import binary_dilation

    random.seed(seed)
    selected_rows = df.sample(n=count, random_state=seed).reset_index(drop=True)

    fig, axes = plt.subplots(count, 3, figsize=(12, 3 * count))
    if count == 1:
        axes = [axes]

    for i, row in selected_rows.iterrows():
        file_name = row['FileName']  # e.g., 0X100009310A3BD7FC_46.png
        coords = row['MaskCoordinates']

        # Full path
        image_path = os.path.join(image_folder, file_name)
        if not os.path.exists(image_path):
            print(f"⚠️ Image not found: {image_path}")
            continue

        # Load image
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(cv2.resize(img, (112, 112)), cv2.COLOR_BGR2RGB)

        # Generate mask
        mask = generate_mask_image(coords, width=112, height=112)

        # Overlay
        neon_orange = np.array([255, 110, 0], dtype=np.uint8)
        overlay = img_rgb.copy()
        # Inner fill
        for c in range(3):
            overlay[:, :, c] = np.where(mask == 1,
                                        (0.25 * neon_orange[c] + 0.65 * overlay[:, :, c]).astype(np.uint8),
                                        overlay[:, :, c])
        # Border
        dilated = binary_dilation(mask, structure=np.ones((3,3)))
        border = (dilated & ~mask).astype(np.uint8)
        for c in range(3):
            overlay[:, :, c] = np.where(border == 1, neon_orange[c], overlay[:, :, c])

        # Plot
        axes[i][0].imshow(img_rgb)
        axes[i][0].set_title("Original")
        axes[i][1].imshow(mask, cmap='gray')
        axes[i][1].set_title("Mask")
        axes[i][2].imshow(overlay)
        axes[i][2].set_title("Overlay")
        for ax in axes[i]:
            ax.axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
plot_combined_mask_overlay(df_existing_masks, frames_dir, count=8)


In [ ]:
save_all_masks(df_mask_list_format_merged, output_mask_dir)


In [ ]:
shutil.make_archive(output_mask_dir, 'zip', output_mask_dir)


In [ ]:
from PIL import Image
import numpy as np

def print_distinct_pixel_values_kaggle_style(image_path):
    # Load image in grayscale
    img = Image.open(image_path).convert('L')

    # Convert to binary mask: 0 stays 0, anything else becomes 1
    mask_array = (np.array(img) > 0).astype(np.uint8)

    unique_vals = np.unique(mask_array)

    print("Grayscale image (binary):")
    print(f"Distinct pixel values ({len(unique_vals)} total):")
    print(unique_vals)

# Example usage
image_path = output_mask_dir + '/0X100009310A3BD7FC_46.png'  # Replace with your file
print_distinct_pixel_values_kaggle_style(image_path)


In [ ]:
from google.colab import files
import shutil



# Download
files.download("/content/echonet_processed/frames.zip")
files.download("/content/echonet_processed/masks.zip")
